# Supervised Finetuning Dataset

语言模型的输入是 token 序列, 对于有监督学习(Supervised FineTuning,SFT)任务可以将语言任务数据转化为通用的问答（QA）模式，SFT学习时仅拟合回答(A)

所以有监督任务数据面临（1）将有监督任务数据转化为通用的 QA (2) 构造有监督学习的 label

1. message format
2. dataset
3. data collactor function

处理流程

1. SFT 数据为 QA 对, 考虑多轮对话情形, 使用 chat message 组织对话，使用自定义对话模版
2. 将单条数据 tokenizer 化
3. 编写 collate function

## 数据 & Tokenizer

In [21]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B', 
                                          local_dir='~/.cache/huggingface/', # 如果可以直连 huggingface, 去除此行
                                         )

In [48]:
DEFINIED_SYSTEM_PROMPT='你是小冬瓜智能体,请安全详细回答用户 USER 的问题'
messages_1=[
    {'role':'SYSTEM', 'content':DEFINIED_SYSTEM_PROMPT},
    {'role':'USER', 'content':'$sin^2x+cos^2x=?'},
    {'role':'ASSISTANT', 'content':'结果为 $\\boxed{1}$'},
    {'role':'USER', 'content':'为什么?'},
    {'role':'ASSISTANT', 'content':'在单位圆上的任意一点与原点的连线,该线在 xy 轴上的投影分别为$sinx,cosx$,根据勾股定理可证$sin^2x+cos^2x=1$'},
]

messages_2=[    
    {'role':'SYSTEM', 'content':DEFINIED_SYSTEM_PROMPT},
    {'role':'USER', 'content':'什么是人工智能?'},
    {'role':'ASSISTANT', 'content':'人工智能是让机器模拟人类思维的技术。'},
]
messages_3=[    
    {'role':'SYSTEM', 'content':DEFINIED_SYSTEM_PROMPT},
    {'role':'USER', 'content':'如何计算复利?'},
    {'role':'ASSISTANT', 'content':'复利计算公式：本息和 = 本金 × (1 + 利率)^期数。'},
]
messages_4=[    
    {'role':'SYSTEM', 'content':DEFINIED_SYSTEM_PROMPT},
    {'role':'USER', 'content':'“哈基米”翻译成英文'},
    {'role':'ASSISTANT', 'content':'“哈基米”翻译成英文通常是 "Hakimi"（人名音译）。'},
]
messages_list = [messages_1, messages_2, messages_3, messages_4]

## Chat Template

对话模版有两种组织方式：

1. 对文本进行格式化处理
2. 对 token id 进行处理

### 方式1

In [49]:
def ChatTemplate(example):
    prompt = '<SOS>'
    for i, item in enumerate(example):
        prompt += '#' + item['role'] + ':' + item['content'] 
        if i % 2 == 0 and i != 0:
            prompt += '<EOS>' # 补全回答都要 EOS
    return prompt
    
prompt = ChatTemplate(messages_1)
print(prompt)

<SOS>#SYSTEM:你是小冬瓜智能体,请安全详细回答用户 USER 的问题#USER:$sin^2x+cos^2x=?#ASSISTANT:结果为 $\boxed{1}$<EOS>#USER:为什么?#ASSISTANT:在单位圆上的任意一点与原点的连线,该线在 xy 轴上的投影分别为$sinx,cosx$,根据勾股定理可证$sin^2x+cos^2x=1$<EOS>


In [50]:
import torch

input_id = tokenizer( [prompt], add_special_tokens=True ) # tokenizer 默认输入列表
print(input_id['input_ids'])

decode_prompt = tokenizer.decode(input_id['input_ids'][0], skip_special_tokens=False)
print(decode_prompt)

[[18858, 3126, 61125, 46487, 25, 105043, 30709, 99949, 100857, 100168, 31914, 11, 14880, 99464, 100700, 102104, 20002, 13872, 43589, 86119, 2, 6448, 21701, 15940, 61, 17, 87, 10, 9407, 61, 17, 87, 19884, 2, 4939, 3846, 2821, 25, 59151, 17714, 57960, 79075, 90, 16, 31716, 27, 55940, 61125, 6448, 25, 100678, 30, 2, 4939, 3846, 2821, 25, 18493, 75317, 100213, 101913, 108112, 100380, 57218, 52129, 27442, 9370, 116539, 11, 75882, 43268, 18493, 30784, 8908, 121, 112, 101913, 111367, 105706, 3, 15940, 87, 11, 9407, 87, 54876, 100345, 105170, 99223, 22382, 21887, 30440, 33477, 3, 15940, 61, 17, 87, 10, 9407, 61, 17, 87, 28, 16, 3, 27, 55940, 29]]
<SOS>#SYSTEM:你是小冬瓜智能体,请安全详细回答用户 USER 的问题#USER:$sin^2x+cos^2x=?#ASSISTANT:结果为 $\boxed{1}$<EOS>#USER:为什么?#ASSISTANT:在单位圆上的任意一点与原点的连线,该线在 xy 轴上的投影分别为$sinx,cosx$,根据勾股定理可证$sin^2x+cos^2x=1$<EOS>


1. 有监督学习任务，目标是拟合 Assistant 内容, 其损失函数与 pre-trained 时使用的 Cross-Entropy Loss 一致
2. 分析以上处理方法，tokenize 字符串时, 就需要在 token id 序列中找到 ASSISTANT 的序列内容

截取回答内容需要定位到:

1. 头: #ASSISTANT 最后一个 token
2. 尾: <EOS>

代码省略在列表查找子列表位置问题

In [57]:
ids = tokenizer('#ASSISTANT')['input_ids']
print(ids)
for i in ids:
    print(tokenizer.decode(i))

ids = tokenizer('<EOS>')['input_ids'] #并非按照我们期望encode成一个 token id
print(ids)
for i in ids:
    print(tokenizer.decode(i)) 

[2, 4939, 3846, 2821]
#
ASS
IST
ANT
[23835, 3126, 29]
<E
OS
>


### Tokenizer 分析

Tokenizer 会预设专用的 token，以 Qwen3 举例, 有 eos_token `<|im_end|>`, 但是句子开头在`'additional_special_tokens':` 上的 `<|im_start|>`

较为特殊的是有 `'pad_token': '<|endoftext|>',`, 对应的词元写法应当为'<|pad|>'更加合适, 本 lc 不进行过度修改

In [61]:
tokenizer.special_tokens_map

{'eos_token': '<|im_end|>',
 'pad_token': '<|endoftext|>',
 'additional_special_tokens': ['<|im_start|>',
  '<|im_end|>',
  '<|object_ref_start|>',
  '<|object_ref_end|>',
  '<|box_start|>',
  '<|box_end|>',
  '<|quad_start|>',
  '<|quad_end|>',
  '<|vision_start|>',
  '<|vision_end|>',
  '<|vision_pad|>',
  '<|image_pad|>',
  '<|video_pad|>']}

In [65]:
DEFINED_EOS_TOKEN = '<|im_end|>'
DEFINED_SOS_TOKEN = '<|im_start|>'
DEFINED_PAD_TOKEN = '<|endoftext|>'

def ChatTemplateDefinedToken(example):
    prompt = DEFINED_SOS_TOKEN
    for i, item in enumerate(example):
        prompt += '#' + item['role'] + ':' + item['content'] 
        if i % 2 == 0 and i != 0:
            prompt += DEFINED_EOS_TOKEN # 补全回答都要 EOS
    return prompt


In [66]:
prompt = ChatTemplateDefinedToken(messages_1)
print(prompt)

input_id = tokenizer( [prompt], add_special_tokens=True ) # tokenizer 默认输入列表
print(input_id['input_ids'])

decode_prompt = tokenizer.decode(input_id['input_ids'][0], skip_special_tokens=False)
print(decode_prompt)

<|im_start|>#SYSTEM:你是小冬瓜智能体,请安全详细回答用户 USER 的问题#USER:$sin^2x+cos^2x=?#ASSISTANT:结果为 $\boxed{1}$<|im_end|>#USER:为什么?#ASSISTANT:在单位圆上的任意一点与原点的连线,该线在 xy 轴上的投影分别为$sinx,cosx$,根据勾股定理可证$sin^2x+cos^2x=1$<|im_end|>
[[151644, 2, 46487, 25, 105043, 30709, 99949, 100857, 100168, 31914, 11, 14880, 99464, 100700, 102104, 20002, 13872, 43589, 86119, 2, 6448, 21701, 15940, 61, 17, 87, 10, 9407, 61, 17, 87, 19884, 2, 4939, 3846, 2821, 25, 59151, 17714, 57960, 79075, 90, 16, 31716, 151645, 2, 6448, 25, 100678, 30, 2, 4939, 3846, 2821, 25, 18493, 75317, 100213, 101913, 108112, 100380, 57218, 52129, 27442, 9370, 116539, 11, 75882, 43268, 18493, 30784, 8908, 121, 112, 101913, 111367, 105706, 3, 15940, 87, 11, 9407, 87, 54876, 100345, 105170, 99223, 22382, 21887, 30440, 33477, 3, 15940, 61, 17, 87, 10, 9407, 61, 17, 87, 28, 16, 3, 151645]]
<|im_start|>#SYSTEM:你是小冬瓜智能体,请安全详细回答用户 USER 的问题#USER:$sin^2x+cos^2x=?#ASSISTANT:结果为 $\boxed{1}$<|im_end|>#USER:为什么?#ASSISTANT:在单位圆上的任意一点与原点的连线,该线在 xy 轴上的投影分别为$sinx,cosx$,

## Dataset

## Data collate function